# 🏆 Fine-Tune Llama 3.1 8B for Customer Support

**Goal:** Fine-tune a model to both classify support tickets (category + urgency) and generate responses.

**Method:** QLoRA via Unsloth on a free Google Colab T4 GPU (16GB VRAM).

---

**⏱ Runtime:** Enable T4 GPU: `Runtime → Change runtime type → T4 GPU`

**💾 Save your work:** Log in to Hugging Face (Step 2) so you can push your model to the Hub.

## 1. Install Dependencies

Unsloth now uses `uv` for fast, reliable installation. This takes ~2 minutes.

In [ ]:
# Install Unsloth using uv (fast package installer)
!pip install uv
!uv pip install unsloth --torch-backend=auto --system

# Additional dependencies
!uv pip install datasets huggingface-hub evaluate bert-score rouge-score scikit-learn --system

print("\n✓ All dependencies installed!")

## 2. Login to Hugging Face (recommended)

This lets you push your fine-tuned model to the Hub so you can use it anywhere.

In [ ]:
from huggingface_hub import notebook_login

# Get your token from: https://huggingface.co/settings/tokens
notebook_login()

# Set your HF username here for saving later
HF_USERNAME = "your-username"  # <-- CHANGE THIS

## 3. Load & Prepare Datasets

We combine three datasets for a dual-task model:
| Dataset | Use | Size |
|---------|-----|------|
| **Bitext Customer Support** | Response generation + intent classification | ~30k |
| **ServiceNow Incidents** | Category + urgency + resolution | ~10k |
| **TriageIQ** | Sentiment + urgency + category | ~5k |

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("Loading datasets...")

# ──── 1. Bitext Customer Support ────
bitext = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(f"  Bitext: {len(bitext):,} samples")

# ──── 2. ServiceNow Incidents (synthetic) ────
servicenow = load_dataset("6StringNinja/synthetic-servicenow-incidents", split="train")
print(f"  ServiceNow: {len(servicenow):,} samples")

# ──── 3. TriageIQ ────
triageiq = None
try:
    triageiq = load_dataset("coldstart88/triageiq-dataset", split="train")
    print(f"  TriageIQ: {len(triageiq):,} samples")
except Exception as e:
    print(f"  TriageIQ: skipped ({e})")

In [ ]:
# ──── Unified Formatting ────

def format_bitext(ex):
    """Bitext → instruction format."""
    instruction = "Classify the following customer support inquiry and provide a helpful response."
    input_text = ex.get('instruction', ex.get('input', '')) or ex.get('query', '')
    category = ex.get('intent', 'general')
    response = ex.get('response', ex.get('output', ''))
    return {
        'instruction': instruction,
        'input': f"Ticket: {input_text}",
        'output': f"Category: {category}\nResponse: {response}",
        'source': 'bitext'
    }

def format_servicenow(ex):
    """ServiceNow → instruction format."""
    instruction = "Classify this support ticket by category and urgency, then provide a resolution."
    description = ex.get('short_description', ex.get('description', ''))
    category = ex.get('category', 'Uncategorized')
    urgency = ex.get('urgency', '3')
    resolution = ex.get('resolution', ex.get('resolution_notes', 'Investigate and resolve.'))
    urgency_map = {'1': 'Critical', '2': 'High', '3': 'Medium', '4': 'Low', '5': 'Low'}
    urgency_label = urgency_map.get(str(urgency).strip(), 'Medium')
    return {
        'instruction': instruction,
        'input': f"Ticket: {description}",
        'output': f"Category: {category}\nUrgency: {urgency_label}\nResolution: {resolution}",
        'source': 'servicenow'
    }

def format_triageiq(ex):
    """TriageIQ → instruction format."""
    instruction = "Analyze the sentiment, urgency, and category of this customer message."
    text = ex.get('text', ex.get('message', ''))
    sentiment = ex.get('sentiment', 'neutral')
    urgency = ex.get('urgency', 'medium')
    category = ex.get('category', 'general')
    return {
        'instruction': instruction,
        'input': f"Message: {text}",
        'output': f"Category: {category}\nUrgency: {urgency}\nSentiment: {sentiment}",
        'source': 'triageiq'
    }

# Apply formatting
all_examples = []

for ex in bitext:
    try:
        all_examples.append(format_bitext(ex))
    except:
        pass

for ex in servicenow:
    try:
        all_examples.append(format_servicenow(ex))
    except:
        pass

if triageiq:
    for ex in triageiq:
        try:
            all_examples.append(format_triageiq(ex))
        except:
            pass

print(f"Total formatted examples: {len(all_examples):,}")

In [ ]:
# ──── Convert to DataFrame, sample, split ────
df = pd.DataFrame(all_examples)
print(f"Sources: {df['source'].value_counts().to_dict()}")

# Limit to 10k for fast training (~1 hour on T4)
MAX_SAMPLES = 10_000
if len(df) > MAX_SAMPLES:
    df = df.sample(MAX_SAMPLES, random_state=3407).reset_index(drop=True)
    print(f"Sampled to {MAX_SAMPLES} examples")

# Split
train_df, eval_df = train_test_split(df, test_size=0.1, random_state=3407)
print(f"Train: {len(train_df):,} | Eval: {len(eval_df):,}")

from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))

## 4. Load Base Model + LoRA (4-bit)

Loading Llama 3.1 8B in 4-bit quantization. This uses ~8GB VRAM, leaving room for training.

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
# Alternative: "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" (slightly smaller, faster)

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    device_map="auto",
)
print("✓ Base model loaded!")

# ──── Add LoRA Adapters ────
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ LoRA adapters added! Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. Tokenize Dataset

Applying Llama 3's chat template so the model learns the instruction-following format.

In [ ]:
def tokenize_function(examples):
    """Tokenize chat-formatted examples."""
    texts = []
    for i in range(len(examples['instruction'])):
        messages = [
            {"role": "system", "content": examples['instruction'][i]},
            {"role": "user", "content": examples['input'][i]},
            {"role": "assistant", "content": examples['output'][i]}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)

    tokenized = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_seq_length,
        return_tensors=None,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizing...")
tokenized_train = train_dataset.map(
    tokenize_function, batched=True,
    remove_columns=train_dataset.column_names, batch_size=32,
)
tokenized_eval = eval_dataset.map(
    tokenize_function, batched=True,
    remove_columns=eval_dataset.column_names, batch_size=32,
)
print(f"✓ Tokenized: {len(tokenized_train)} train + {len(tokenized_eval)} eval")

## 6. Train!

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./results/llama-support",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,       # Effective batch size = 8
    warmup_steps=5,
    max_steps=500,                        # ~1-1.5 hours on T4
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    report_to="none",
    ddp_find_unused_parameters=False,
    gradient_checkpointing=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    args=training_args,
    max_seq_length=max_seq_length,
    # Dataset is pre-tokenized (has input_ids, labels), so no dataset_text_field needed
)

print("Starting training... (this takes ~1 hour)")
trainer.train()
print("\n✓ Training complete!")

## 7. Save & Push to Hugging Face Hub

In [ ]:
# Save locally (in Colab)
model.save_pretrained("./llama-support-lora")
tokenizer.save_pretrained("./llama-support-lora")
print("✓ Saved locally to ./llama-support-lora")

# Push to Hugging Face Hub (make sure you're logged in from Step 2)
if HF_USERNAME != "your-username":
    repo_name = f"{HF_USERNAME}/llama-support-lora"
    model.push_to_hub(repo_name)
    tokenizer.push_to_hub(repo_name)
    print(f"✓ Pushed to Hugging Face: {repo_name}")
    print(f"  Anyone can use it with: FastLanguageModel.from_pretrained('{repo_name}')")
else:
    print("⏸ Skipped pushing to Hub (set HF_USERNAME above to enable)")

## 8. Quick Test

Let's see how the fine-tuned model handles real tickets.

In [ ]:
FastLanguageModel.for_inference(model)

test_tickets = [
    "My internet has been down for 3 hours and I have an important meeting in 30 minutes. This is urgent!",
    "How do I reset my password? I forgot it after the last update.",
    "I was charged twice for my subscription this month. Please refund the extra charge.",
]

for ticket in test_tickets:
    messages = [
        {"role": "system", "content": "Classify this support ticket by category and urgency, then generate a response."},
        {"role": "user", "content": f"Ticket: {ticket}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    print(f"\n{'='*70}")
    print(f"📝 TICKET: {ticket}")
    print(f"💬 RESPONSE:\n{response}")

print(f"\n{'='*70}")
print("✅ Inference looks good!")

## 9. Save to Google Drive (backup)

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/llama-support-lora'
if os.path.exists(drive_path):
    shutil.rmtree(drive_path)
shutil.copytree('./llama-support-lora', drive_path)
print(f"✓ Backed up to: {drive_path}")

---

## 🎉 You're Done!

**Next steps after training:**
1. Run `python src/eval/benchmark.py --finetuned-model ./llama-support-lora` to get before/after metrics
2. Wire the model into the MCP server: `python src/mcp_server/server.py --transport cli`
3. Import the n8n workflows and connect them to the MCP server
4. (Optional) DPO alignment pass for better response quality